##Akuna Capital - 2025 Options Market Making - Quant Trading Competition (Rank: Top 10)

**Author:** Saurav Luthra  
**Date:** August 2025

In this options market marking competition the task was to implement a custom `MarketMaker` algorithm to continuosly generate competitive bid/ask quotes across European-style options chains in a simulated trading environment against other market making bots. At each timestep, the bot received a set of underlying assets and a list of tradeable option contracts available in the market.

Using a custom option pricer and spread model, the algorithm price options, maintained a delta-hedged inventory, and computed option sensitivities to appropriately model the bid/ask spread.


**Trading Environment Classes:**

- `Option`:
    - Defines the option contract (K, T, option_type, option_id, underlying_id)
- `OptionType`:
    - Defines type of an option (calls vs puts).
- `Underlying`:
    - Defines the asset underlying an option (S, u, d, p, q, stdev)
- `Position`:
    - Defines the current options and underlying positions/inventory.
- `BaseMarketMaker`:
    - An abstract class that we extend to `MarketMaker`
- `MarketMaker`:
    - Pricing options  
    - Generating Bid/Ask Quotes
    - Delta Hedging  
    - Computing Greeks  


**Mathematical Model:**

The underlying, $S$, followed a binomial random walk:

- At each step, the price moves up by $+u$ or down by $–d$
- Drift-free condition (martingale): $S_t = \mathbb{E}[S_{t+1}] = \mathbb{E}[S_{t+2}] = \space ...$

$$
S_{t+1} =
\begin{cases}
S_t + u, & \text{with probability } p \\\\
S_t - d, & \text{with probability } q = 1 - p
\end{cases}
$$

European-style options can only be exercised at expiry $(t = T)$, meaning the option's payoff is a function of the terminal underlying price, $S_T$. Since the underlying is a recombining binomial tree, we have $T + 1$ possible prices at expiry. The option's current price is equal to its expected payoff at time T:

$$
P_{call} = \mathbb{E} [max(S_T - K, 0)]
$$

$$
P_{put} = \mathbb{E} [max(K - S_T, 0)]
$$

To model the distribution of $S_T$, we create a vector of all possible final prices  (excluding noise),

$$
\mathbf{S_T} = \begin{bmatrix}
s_T^0 \\
s_T^1 \\
\vdots \\
s_T^{T}\
\end{bmatrix}
$$

$$
S_T^{n_u} = S_t + n_u \cdot u - (T - n_u) \cdot d, \space n_{u} = 0,1,...,T
$$

and compute the probability of each outcome using the binomial distribution:
$$
P(S_T = S_T^{n_u}) = C(T, n_u) \cdot p^{n_u} \cdot (1-p)^{T-n_u}
$$

In addition, each time step has an added Gaussian noise term with known constant parameters. Simulating the noise at every step leads to exponential growth of paths (branching into many thousands of scenarios). However, since Gaussians add linearly, the total sum of noise terms over T steps can be simplified to one term. This greatly reduce the computational complexity by adding a single Gaussian sample at time T to the terminal underlying price from above:

$$
\epsilon_t \sim N(0, \space \sigma^2)
$$

$$
\epsilon_{total} = \sum_{t=1}^T \epsilon_t \sim N(0, \space T \cdot \sigma^2)
$$

$$
\sigma_{total} = \sqrt{(T \cdot \sigma^2)}
$$

$$
S_{T + \epsilon} = S_T \space + \epsilon_{total}
$$

We can construct a sampled vector of the total noise term, and apply it to all of the terminal prices in vector $S_T$, find the payoff under that realization, and then take a probability-weighted sum of all the option payoffs. This will yield the option price:


## MarketMaker Implementation

The `MarketMaker` class handled:  
1. **Option Pricing**  
   - Simulate binomial tree terminal payoffs with or without noise, using 1000 equally spaced quantiles.

2. **Quote Generation**  
   - Bid/ask spreads widened in proportion to risk
   - **Next steps:** quote bid/ask in terms of a fixed volatility spread, which would require a new function that can price a given option at a different volatility level.
   - **Next steps:** skew the spread around the calculated fair price according to inventory  

3. **Option Sensitivities (Greeks)**  
   - Delta, Gamma, Theta were computed using finite difference approximations. The options purchased/sold were always delta hedged in the `on_bid_hit` and `on_offer_hit` classes, however, higher gamma options have more risk of the hedge being insufficient to replicate the portfolio due to curvature.
   - These values were used to scale spreads (higher Gamma/Theta → wider spreads, etc.)

**Important point:**

Although $\mathbb{E}[\epsilon_{total}] = 0$, we **cannot ignore it**. Option payoffs are **nonlinear and convex functions** of the underlying price. Due to this convexity, the expected option payoff under noise is always **higher** than under a noiseless binomial tree. Including it is essential for correct valuation.


In [ ]:
import abc
import random
from collections import defaultdict
from collections.abc import Callable
from dataclasses import dataclass, replace
from enum import Enum, auto
from typing import Any
import math
from scipy.stats import norm
import numpy as np

class OptionType(Enum):
    CALL = auto()
    PUT = auto()

    def __str__(self) -> str:
        return "C" if self is OptionType.CALL else "P"


@dataclass(eq=True, frozen=True, unsafe_hash=True)
class Option:
    option_id: int
    option_type: OptionType
    steps_until_expiry: int
    strike: int
    underlying_id: int
    underlying_name: str

    def __post_init__(self) -> None:
        if self.steps_until_expiry < 0:
            raise ValueError("Steps until expiry must be non-negative")

    def __str__(self) -> str:
        return f"{self.option_id} ({self.steps_until_expiry}s {self.underlying_name} {self.strike}{self.option_type!s})"

    @classmethod
    def from_underlying(
        cls: type["Option"],
        underlying: "Underlying",
        option_id: int,
        option_type: OptionType,
        steps_until_expiry: int,
        strike: int,
    ) -> "Option":
        return Option(
            option_id=option_id,
            option_type=option_type,
            steps_until_expiry=steps_until_expiry,
            strike=strike,
            underlying_id=underlying.underlying_id,
            underlying_name=underlying.name,
        )

    def advance_step(self) -> "Option":
        if self.steps_until_expiry == 0:
            return self

        return replace(self, steps_until_expiry=self.steps_until_expiry - 1)

    def contract_matches(self, other: "Option") -> bool:
        return replace(other, option_id=self.option_id) == self

    def expiry_valuation(self, underlying_valuation: float) -> float:
        if self.option_type == OptionType.CALL:
            return max(0, underlying_valuation - self.strike)
        return max(0, self.strike - underlying_valuation)


class Position:
    def __init__(self) -> None:
        self.option_quantity_by_option_id: dict[int, int] = defaultdict(int)
        self.underlying_quantity_by_underlying_id: dict[int, float] = defaultdict(float)

    def add_option_quantity(self, option_id: int, quantity: int) -> None:
        self.option_quantity_by_option_id[option_id] += quantity

    def add_underlying_quantity(self, underlying_id: int, quantity: float) -> None:
        quantity = round(quantity, 2)
        self.underlying_quantity_by_underlying_id[underlying_id] += quantity


@dataclass(frozen=True)
class Underlying:
    name: str
    underlying_id: int

    valuation: float

    down_move_probability: float
    down_move_step: float
    noise_std_dev: float
    up_move_probability: float
    up_move_step: float

    def __eq__(self, other: Any) -> bool:
        if not isinstance(other, Underlying):
            return False
        return self.underlying_id == other.underlying_id

    def __post_init__(self) -> None:
        if self.down_move_step <= 0 or self.up_move_step <= 0:
            raise ValueError("Down/up move steps must both be positive")

        if self.down_move_probability <= 0 or self.up_move_probability <= 0:
            raise ValueError("Down/up move probabilities must both be positive")

        if self.down_move_probability + self.up_move_probability != 1:
            raise ValueError("Down and up move probabilities must sum to 1")

        if ((self.down_move_probability * self.down_move_step) - (self.up_move_probability * self.up_move_step)) > 1e-5:
            raise ValueError("Underlying has drift")

    def advance_step(self) -> "Underlying":
        if random.random() < self.up_move_probability:
            valuation: float = self.valuation + self.up_move_step
        else:
            valuation = self.valuation - self.down_move_step

        valuation += random.gauss(sigma=self.noise_std_dev)
        valuation = max(valuation, 0)
        return replace(self, valuation=round(valuation, 2))


class BaseMarketMaker:
    def __init__(self, underlying_initial_state: list[Underlying], option_initial_state: list[Option]) -> None:
        self.underlying_state: list[Underlying] = underlying_initial_state
        self.active_option_state: list[Option] = option_initial_state
        self.position: Position = Position()

    @property
    @abc.abstractmethod
    def name(self) -> str: ...

    def buy_underlying(self, underlying_id: int, quantity: float) -> None:
        if quantity <= 0:
            raise ValueError("Trade quantity must be positive")

        self.trade_underlying_callback(underlying_id, quantity)
        self.position.add_underlying_quantity(underlying_id, quantity)

    @abc.abstractmethod
    def make_market(self, option: Option) -> tuple[float, float]: ...

    def on_bid_hit(self, option: Option, bid_price: float) -> None:
        self.position.add_option_quantity(option.option_id, 1)

    def on_offer_hit(self, option: Option, offer_price: float) -> None:
        self.position.add_option_quantity(option.option_id, -1)

    def on_step_advance(self, new_underlying_state: list[Underlying], new_option_state: list[Option]) -> None:
        self.underlying_state = new_underlying_state
        self.active_option_state = new_option_state

    def register_trade_underlying_callback(self, trade_underlying_callback: Callable[[int, float], None]) -> None:
        self.trade_underlying_callback = trade_underlying_callback

    @abc.abstractmethod
    def price_option(self, option: Option) -> float: ...

    def sell_underlying(self, underlying_id: int, quantity: float) -> None:
        if quantity <= 0:
            raise ValueError("Trade quantity must be positive")

        self.trade_underlying_callback(underlying_id, -quantity)
        self.position.add_underlying_quantity(underlying_id, -quantity)


class MarketMaker(BaseMarketMaker):
    @property
    def name(self) -> str:
        return "Saurav Luthra"

    # ======== INIT ========
    def __init__(self, underlying_initial_state: list[Underlying], option_initial_state: list[Option]) -> None:
        super().__init__(underlying_initial_state, option_initial_state)
        self.underlying_dict = {u.underlying_id: u for u in underlying_initial_state}
        self.option_dict = {o.option_id: o for o in option_initial_state}

    # ======== ON STEP ADVANCE ========
    def on_step_advance(self, new_underlying_state: list[Underlying], new_option_state: list[Option]) -> None:
        super().on_step_advance(new_underlying_state, new_option_state)
        self.underlying_dict = {u.underlying_id: u for u in new_underlying_state}
        self.option_dict = {o.option_id: o for o in new_option_state}
        self.delta_hedge()

    # ======== ON BID HIT ========
    def on_bid_hit(self, option: Option, bid_price: float) -> None:

        super().on_bid_hit(option, bid_price)

        option_delta = abs(self.option_delta(option))
        if str(option.option_type) == 'C' and option_delta > 0:
            self.sell_underlying(option.underlying_id, option_delta)
        elif str(option.option_type) == 'P' and option_delta > 0:
            self.buy_underlying(option.underlying_id, option_delta)

    # ======== ON ASK HIT ========
    def on_offer_hit(self, option: Option, offer_price: float) -> None:

        super().on_offer_hit(option, offer_price)

        option_delta = abs(self.option_delta(option))
        if str(option.option_type) == 'C' and option_delta > 0:
            self.buy_underlying(option.underlying_id, option_delta)
        elif str(option.option_type) == 'P' and option_delta > 0:
            self.sell_underlying(option.underlying_id, option_delta)

    # ======== PRICE OPTION ========
    def price_option(self, option: Option) -> float:

        return self.pricer(option, True)

    # ======== MAKE MARKET ========
    def make_market(self, option: Option) -> tuple[float, float]:

        theo_px = self.pricer(option, True)
        bid_spread, ask_spread = self.bid_ask_spread(option, theo_px)

        return max(theo_px - bid_spread, 0.00), max(theo_px + ask_spread, 0.01)

    # ======== BID-ASK SPREAD ========
    def bid_ask_spread(self, option: Option, theo_px: float) -> tuple[float, float]:

        underlying  = self.underlying_dict[option.underlying_id]

        risk_factor = 0.5
        gamma       = self.option_gamma(option)
        gamma2      = gamma ** 2
        theta2      = self.option_theta(option) ** 2
        sigma2      = underlying.noise_std_dev ** 2
        T           = option.steps_until_expiry

        spread2     = risk_factor * math.sqrt((gamma2 * sigma2 * T) + (theta2 * T))
        return spread2 * 0.5, spread2 * 0.5

    # ======== DELTA HEDGE ========
    def delta_hedge(self) -> None:

        tgt_posn_by_underlying_id = {underlying_id: 0.0 for underlying_id in self.underlying_dict.keys()}

        for option_id, option_qty in self.position.option_quantity_by_option_id.items():
            if option_id in self.option_dict.keys():
                option = self.option_dict[option_id]
                delta  = self.option_delta(option)
                tgt_posn_by_underlying_id[option.underlying_id] -= delta * option_qty

        for underlying_id, tgt_posn in tgt_posn_by_underlying_id.items():
            buy_sell_qty = tgt_posn - self.position.underlying_quantity_by_underlying_id[underlying_id]
            if buy_sell_qty > 0:
                self.buy_underlying(underlying_id, buy_sell_qty)
            elif buy_sell_qty < 0:
                self.sell_underlying(underlying_id, abs(buy_sell_qty))

    # ======== PRICER ========
    def pricer(self, option: Option, model_shocks: bool, S_input: float = -1) -> float:

        underlying = self.underlying_dict[option.underlying_id]

        S0  = underlying.valuation
        if S_input >= 0:
            S0 = S_input
        T   = option.steps_until_expiry
        K   = option.strike
        u   = underlying.up_move_step
        p   = underlying.up_move_probability
        d   = underlying.down_move_step
        q   = underlying.down_move_probability
        sd  = underlying.noise_std_dev

        n = 1000
        probs = np.linspace(0.0001, 0.9999, n)
        if model_shocks and sd > 0:
            epsilon_total = norm.ppf(probs, loc = 0, scale = (math.sqrt(T) * sd))
        else:
            epsilon_total = np.zeros(n)

        theo_px = 0
        for ups in range(0, T + 1):

            ST_prob     = math.comb(T, ups) * (p ** ups) * (q ** (T - ups))
            ST_val      = max(S0 + (u * ups) - (d * (T - ups)), 0)

            if str(option.option_type) == 'C':
                payouts = (ST_val - K) + epsilon_total
            else:
                payouts = (K - ST_val) + epsilon_total

            EV_payouts  = np.sum(payouts[payouts > 0]) * (1/n)
            theo_px    += EV_payouts * ST_prob

        return round(float(theo_px), 4)

    # ======== OPTION DELTA ========
    def option_delta(self, option: Option) -> float:

        if option.steps_until_expiry < 1:
            return 0

        underlying = self.underlying_dict[option.underlying_id]
        S0 = underlying.valuation
        u  = underlying.up_move_step
        d  = underlying.down_move_step

        option_T1 = option.advance_step()

        Su = S0 + u
        Sd = S0 - d

        Vu = self.pricer(option_T1, True, Su)
        Vd = self.pricer(option_T1, True, Sd)

        return (Vu - Vd) / (Su - Sd)

    # ======== OPTION GAMMA ========
    def option_gamma(self, option: Option) -> float:

        if option.steps_until_expiry < 3:
            return 1

        underlying = self.underlying_dict[option.underlying_id]
        S0 = underlying.valuation
        u  = underlying.up_move_step
        d  = underlying.down_move_step

        option_T1 = option.advance_step()
        option_T2 = option_T1.advance_step()

        Su  = S0 + u
        Sd  = S0 - d
        Suu = S0 + u + u
        Sud = S0 + u - d
        Sdu = S0 - d + u
        Sdd = S0 - d - d

        Vuu = self.pricer(option_T2, True, Suu)
        Vud = self.pricer(option_T2, True, Sud)
        Vdu = self.pricer(option_T2, True, Sdu)
        Vdd = self.pricer(option_T2, True, Sdd)

        Du = (Vuu - Vud) / (Suu - Sud)
        Dd = (Vdu - Vdd) / (Sdu - Sdd)

        return (Du - Dd) / (Su - Sd)

    # ======== OPTION THETA ========
    def option_theta(self, option: Option) -> float:

        if option.steps_until_expiry < 1:
            return 0

        underlying = self.underlying_dict[option.underlying_id]
        S0 = underlying.valuation

        option_T1 = option.advance_step()

        V0 = self.pricer(option, True, S0)
        V1 = self.pricer(option_T1, True, S0)

        return (V1 - V0)
